# 🌿 Hierarchical Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> Hierarchical clustering builds a tree of clusters — you can explore any number of segments by cutting the dendrogram at different heights. Unlike K-means, you don't need to specify K upfront.

### Dataset
**S&P 500 Stock Returns — Sector Clustering**
We cluster major US stocks by their daily return correlations. Stocks that move together get clustered together — revealing sector structure and diversification opportunities. This is how portfolio managers and risk analysts actually use clustering.

### Time: ~55 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import subprocess, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# Clone course repo to get bundled datasets
if not os.path.exists('pattern-recognition-notebooks'):
    print("Downloading course data...")
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '--quiet',
         'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"Clone failed: {result.stderr[:200]}")
    else:
        print("✓ Course data ready")
else:
    print("✓ Course data already present")

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'  # fallback for local use

print(f"  Data folder: {DATA_DIR}")
print(f"  Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

# Technique-specific imports
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


In [ ]:
# Real S&P 500 stock returns via yfinance
# yfinance is pre-installed in Colab and pulls live data from Yahoo Finance
!pip install yfinance -q

import yfinance as yf

stocks = {
    'Tech':       ['AAPL','MSFT','GOOGL','META','NVDA','AMZN'],
    'Finance':    ['JPM','BAC','GS','MS','WFC','C'],
    'Healthcare': ['JNJ','PFE','UNH','ABT','MRK','TMO'],
    'Energy':     ['XOM','CVX','COP','SLB','PSX','VLO'],
    'Consumer':   ['PG','KO','PEP','WMT','COST','MCD'],
}

all_tickers   = [t for tickers in stocks.values() for t in tickers]
sector_labels = {t: s for s, tickers in stocks.items() for t in tickers}

# Pull 2 years of daily closing prices (2022-2024 — full market cycle inc. downturn + recovery)
print("Downloading real S&P 500 price data from Yahoo Finance...")
prices = yf.download(all_tickers, start='2022-01-01', end='2024-01-01',
                     auto_adjust=True, progress=False)['Close']

# Drop any tickers that failed to download
prices = prices.dropna(axis=1, how='all')
available = [t for t in all_tickers if t in prices.columns]
missing   = [t for t in all_tickers if t not in prices.columns]
if missing:
    print(f"  Note: {missing} unavailable — continuing with {len(available)} tickers")

# Daily log returns (more statistically appropriate than simple returns)
returns = np.log(prices[available] / prices[available].shift(1)).dropna()

# Update sector labels to only include available tickers
sector_labels = {t: s for t, s in sector_labels.items() if t in available}
all_tickers   = available

print(f"\nReal S&P 500 returns — {len(all_tickers)} stocks")
print(f"Date range: {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Trading days: {len(returns)}")
print(f"\nSectors: {list(stocks.keys())}")
print(f"\nAvg daily return:     {returns.mean().mean()*100:.3f}%")
print(f"Avg daily volatility: {returns.std().mean()*100:.2f}%")
print(f"\nSample correlation (AAPL vs MSFT): {returns['AAPL'].corr(returns['MSFT']):.3f}")
print(f"Sample correlation (AAPL vs XOM):  {returns['AAPL'].corr(returns['XOM']):.3f}")
print("(Tech-Tech correlation should be higher than Tech-Energy)")


## 📐 Part 1 — Clustering Stocks by Return Correlation

Instead of clustering raw return values, we cluster by **correlation distance**:
```
distance(i,j) = 1 - correlation(i,j)
```
Stocks that move together (high correlation) have low distance and cluster together.
This is the standard approach in portfolio risk analysis.

In [ ]:
# Compute correlation matrix and convert to distance
corr_matrix = returns.corr()
dist_matrix = 1 - corr_matrix

# Hierarchical clustering — complete linkage
Z = linkage(squareform(dist_matrix.values), method='complete')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 5))
sector_colors = {'Tech':'#1e5fa8','Finance':'#e85d20','Healthcare':'#1a7a45',
                 'Energy':'#6b46c1','Consumer':'#0e7490'}
leaf_colors = [sector_colors[sector_labels[t]] for t in all_tickers]

dend = dendrogram(Z, labels=all_tickers, ax=ax, leaf_rotation=45,
                  color_threshold=0.7*max(Z[:,2]))
ax.set_title('Hierarchical Clustering of S&P 500 Stocks by Return Correlation\n'
             'Height = dissimilarity — stocks that merge early are most similar')
ax.set_ylabel('Correlation Distance (1 - correlation)')

# Add sector color legend
for sector, color in sector_colors.items():
    ax.plot([], [], 's', color=color, label=sector, markersize=8)
ax.legend(title='Actual Sector', loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()
print("\n\u2714 Stocks within the same sector cluster together at low height")
print("   Energy separates early (negative market correlation)")
print("   This confirms the sector structure without using any sector labels")

In [ ]:
# Cut dendrogram at different heights = different numbers of clusters
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_clusters in zip(axes, [3, 5, 7]):
    cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')

    # Show correlation heatmap colored by cluster
    idx_sorted = np.argsort(cluster_labels)
    sorted_corr = corr_matrix.iloc[idx_sorted, idx_sorted]
    sorted_tickers = [all_tickers[i] for i in idx_sorted]

    im = ax.imshow(sorted_corr.values, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(sorted_tickers)))
    ax.set_yticks(range(len(sorted_tickers)))
    ax.set_xticklabels(sorted_tickers, rotation=90, fontsize=7)
    ax.set_yticklabels(sorted_tickers, fontsize=7)
    ax.set_title(f'{n_clusters} Clusters\n(sorted by cluster membership)')

    # Add cluster boundaries
    boundaries = np.where(np.diff(cluster_labels[idx_sorted]))[0] + 0.5
    for b in boundaries:
        ax.axhline(b, color='black', lw=1.5)
        ax.axvline(b, color='black', lw=1.5)

plt.colorbar(im, ax=axes[-1], label='Correlation', shrink=0.8)
plt.suptitle('Correlation Heatmaps at Different Cluster Cuts\n'
             'Green = high correlation, Red = negative correlation', fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Portfolio diversification insight
n_clusters = 5
cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
stock_clusters = pd.DataFrame({'Ticker': all_tickers,
                                'Sector': [sector_labels[t] for t in all_tickers],
                                'Cluster': cluster_labels})

print("=== Cluster Composition ===\n")
for cl in sorted(stock_clusters['Cluster'].unique()):
    members = stock_clusters[stock_clusters['Cluster']==cl]
    print(f"Cluster {cl} ({len(members)} stocks):")
    for _, row in members.iterrows():
        print(f"  {row['Ticker']:<8} [{row['Sector']}]")
    # Within-cluster avg correlation
    tickers = members['Ticker'].tolist()
    if len(tickers) > 1:
        avg_corr = corr_matrix.loc[tickers, tickers].values
        np.fill_diagonal(avg_corr, np.nan)
        print(f"  Avg within-cluster correlation: {np.nanmean(avg_corr):.3f}")
    print()

print("\u2714 Portfolio insight: pick ONE stock per cluster for maximum diversification")
print("   Stocks in the same cluster move together — owning multiple adds little diversification benefit")

In [ ]:
# @title 📝 Quiz — Hierarchical Clustering
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key advantage of hierarchical over K-means clustering?
# @markdown **Q2:** What does complete linkage measure when merging two clusters?
# @markdown **Q3:** Why do we cluster by correlation rather than raw return levels?
# @markdown **Q4:** How does a dendrogram help choose the number of clusters?
# @markdown **Q5:** If two stocks are in the same cluster, what does that mean for portfolio diversification?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="Hierarchical Clustering"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*